In [1]:
import pandas as pd
import re, unicodedata

In [18]:
path_to_candidate = "data/AI_bert_subs_seed_27_strict.csv"
path_to_labels = "data/AI_bert_subs_seed_27_strict_topics.csv"
path_to_new_csv = "data/subs_labelled_mapped.csv"

In [4]:
df = pd.read_csv(path_to_candidate, index_col=0)
df.head()

,program,year,text_clean,ai_related,matched_keywords_all,relevant_section,processed,nouns,adjectives,verbs,topic,probability
filename,,,,,,,,,,,,
"2016-11-16-19,00-1",de wereld draait door,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"['facebook', 'google', 'twitter', 'algoritme']",...uit Amerika.\nVanochtend geland.\nHet Ameri...,documentaire voetspor undateables seizoen rufu...,documentaire voetspor undateables seizoen rufu...,mooi lang nieuw nepnieuws ander voorafgaand Am...,landen zien maken hebben zeggen willen tegenga...,6,0.102652
"2016-02-08-23,04-1",jinek,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,['drones'],...over.\nEn onze tennisdames verrasten de wer...,tennisdame wereld baan klap sensatie nieuws po...,tennisdame wereld baan klap sensatie nieuws po...,regelrecht ander vorig Nederlands internationa...,verrasten meppen beslissen winnen bekennen opl...,0,0.136302
GOEDEMORGEN_N-WON02434179,goedemorgen nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"['ai', 'kunstmatige intelligentie']",...en de neus van de politie.\nWeet u hoe een ...,neus politie drugslab mens aanraking drug init...,neus politie drugslab mens aanraking drug init...,goed bewuster kunstmatig welmoed hoog laag dir...,weten werken weten denken komen vinden maken z...,19,1.000000
GOEDEMORGEN_N-WON02298553,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"['twitter', 'kunstmatige intelligentie']",...zegt de Britse minister van buitenlandse za...,minister zaak aanval aanval gemeenschap aanval...,minister zaak aanval aanval gemeenschap aanval...,Brits buitenlands roekeloos serieus internatio...,zeggen zeggen nemen doen nemen doen veroordele...,1,0.099202
GOEDEMORGEN_N-WON02108661,goedemorgen nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,['drone'],...er aardig uit.\nIs het een quarantainecoup?...,quarantainecoup filmp laag respect kappersvak ...,quarantainecoup filmp laag respect kappersvak ...,aardig diep moeilijk heel vast flink droog fli...,gekeken proberen opkrijgen doen nemen willen b...,5,1.000000


In [312]:
# #inspect topic probabilities
# print("Min probability :", df['probability'].min())
# print("Max probability :", df['probability'].max())
# print("Mean probability:", df['probability'].mean())
# print("Std deviation   :", df['probability'].std())


In [6]:
def norm_topic(x):
    """Canonicalise topic ids for safe matching across df and Excel."""
    if pd.isna(x):
        return None
    s = str(x)
    # unify unicode + strip invisible/space noise
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", "", s)

    # if it looks numeric, coerce to a canonical string:
    #  - 10.0 -> "10"
    #  - 10.50 -> "10.5"
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        else:
            # normalise trimming trailing zeros/dot
            s2 = ("%.15g" % f)  # compact float repr without scientific if possible
            return s2
    except ValueError:
        # non-numeric topic ids like 'FD50' are kept as-is (case-sensitive)
        return s.strip()

# --- 1) Load mapping and normalise its index ---
mapping = pd.read_csv(path_to_labels, index_col=0)
mapping.index = mapping.index.map(norm_topic)

# Optionally: keep only rows that actually have a label
mapping_labeled = mapping[~mapping["Label NL"].isna()].copy()

# --- 2) Normalise df topics too ---
df["topic_norm"] = df["topic"].map(norm_topic)

# --- 3) Build maps and assign ---
label_map_nl = mapping_labeled["Label NL"]
meta_map_nl  = mapping_labeled["Meta NL"]

label_map = mapping_labeled["Label"]
meta_map  = mapping_labeled["Meta"]

df["topic_label_nl"] = df["topic_norm"].map(label_map_nl)
df["topic_meta_nl"]  = df["topic_norm"].map(meta_map_nl)

df["topic_label"] = df["topic_norm"].map(label_map)
df["topic_meta"]  = df["topic_norm"].map(meta_map)

# --- 4) (Optional) sanity checks ---
# What fraction matched?
matched_frac = df["topic_label"].notna().mean()
print(f"Matched labels for {matched_frac:.1%} of rows.")

# If you want to see why some didn’t match, compare key sets:
missing_keys = set(df["topic_norm"].dropna().unique()) - set(mapping_labeled.index)
if missing_keys:
    print(f"{len(missing_keys)} df topic ids not found in mapping (showing up to 20):",
          sorted(list(missing_keys))[:20])

# --- 5) Keep only labelled rows (discard unlabelled) ---
df_labeled = df[df["topic_label"].notna()].copy()
# If you’re done with the helper column:
# df_labeled.drop(columns=["topic_norm"], inplace=True)


Matched labels for 100.0% of rows.


In [9]:
# show rows where topic_label is 'NOISE'
noise_rows = df_labeled[df_labeled['topic_label'] == 'NOISE']
print(f"Rows labeled as 'NOISE': {len(noise_rows)}")

# show row numbers of rows labeled as 'NOISE'
print("Row numbers of 'NOISE' rows:", noise_rows.index.tolist())

# print one row of 'NOISE' rows to inspect
if not noise_rows.empty:
    print("Example 'NOISE' row:")
    print(noise_rows.iloc[0])
    


Rows labeled as 'NOISE': 0
Row numbers of 'NOISE' rows: []


In [315]:
# df.head()

In [10]:
df = df.dropna(subset=['topic_label']).copy()
df = df[df['topic_meta'] != 'NOISE'].reset_index(drop=True)

print(f"rows dropped due to missing labels: {len(df) - len(df_labeled)}")




rows dropped due to missing labels: -6


In [8]:
df.head()

,program,year,text_clean,ai_related,matched_keywords_all,relevant_section,processed,nouns,adjectives,verbs,topic,probability,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta
0,de wereld draait door,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"['facebook', 'google', 'twitter', 'algoritme']",...uit Amerika.\nVanochtend geland.\nHet Ameri...,documentaire voetspor undateables seizoen rufu...,documentaire voetspor undateables seizoen rufu...,mooi lang nieuw nepnieuws ander voorafgaand Am...,landen zien maken hebben zeggen willen tegenga...,6,0.102652,6,journalisme en media,"MEDIA, KUNST, CULTUUR & SPORT",journalism and media,"MEDIA, ARTS, CULTURE & SPORTS"
1,jinek,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,['drones'],...over.\nEn onze tennisdames verrasten de wer...,tennisdame wereld baan klap sensatie nieuws po...,tennisdame wereld baan klap sensatie nieuws po...,regelrecht ander vorig Nederlands internationa...,verrasten meppen beslissen winnen bekennen opl...,0,0.136302,0,oorlog en politiek,AI OOORLOGSVOERING EN MILITAIRE CONFLICTEN,war and politics,AI WARFARE & MILIRTARY CONFLICTS
2,goedemorgen nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"['ai', 'kunstmatige intelligentie']",...en de neus van de politie.\nWeet u hoe een ...,neus politie drugslab mens aanraking drug init...,neus politie drugslab mens aanraking drug init...,goed bewuster kunstmatig welmoed hoog laag dir...,weten werken weten denken komen vinden maken z...,19,1.000000,19,gezondheid en wetenschap,WETENSCHAP EN GEZONDHEID,health and science,SCIENCE & HEALTH
3,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"['twitter', 'kunstmatige intelligentie']",...zegt de Britse minister van buitenlandse za...,minister zaak aanval aanval gemeenschap aanval...,minister zaak aanval aanval gemeenschap aanval...,Brits buitenlands roekeloos serieus internatio...,zeggen zeggen nemen doen nemen doen veroordele...,1,0.099202,1,publieke discours en maatschappelijke trends,MAATSCHAPPIJ,public discourse and societal trends,SOCIETY
4,goedemorgen nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,['drone'],...er aardig uit.\nIs het een quarantainecoup?...,quarantainecoup filmp laag respect kappersvak ...,quarantainecoup filmp laag respect kappersvak ...,aardig diep moeilijk heel vast flink droog fli...,gekeken proberen opkrijgen doen nemen willen b...,5,1.000000,5,film industrie,"MEDIA, KUNST, CULTUUR & SPORT",film industry,"MEDIA, ARTS, CULTURE & SPORTS"


In [318]:
# overwrite robots & educatie (3) to 1) robots adn 2) tech & ai ontwikkeling

In [12]:
df['topic_meta'].value_counts()

topic_meta
TECH & AI DEVELOPMENT               116
AI WARFARE & MILIRTARY CONFLICTS     98
POLITICS & LAW                       78
MEDIA, ARTS, CULTURE & SPORTS        62
SCIENCE & HEALTH                     28
SOCIETY                              28
EDUCATION                            26
SOCIAL MEDIA                         24
AI FOR SECURITY                      15
BUSINESS & FINANCE                    7
CONSUMER TECH  & PRODUCTS             7
GEOPOLITICS                           6
Name: count, dtype: int64

In [ ]:

mask = (
    (df["topic_label"] == "robots en educatie") &
    (df["topic_meta"] == "EDUCATIE")
)

df.loc[mask, "topic_label"] = "robots"
df.loc[mask, "topic_meta"] = "TECH & AI ONTWIKKELING"

In [321]:
# df.shape

In [19]:
df.to_csv(path_to_new_csv)

In [15]:
df.drop(columns=['topic_label_nl', 'topic_meta_nl'], inplace=True)

In [16]:
df

,program,year,text_clean,ai_related,matched_keywords_all,relevant_section,processed,nouns,adjectives,verbs,topic,probability,topic_norm,topic_label,topic_meta
0,de wereld draait door,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"['facebook', 'google', 'twitter', 'algoritme']",...uit Amerika.\nVanochtend geland.\nHet Ameri...,documentaire voetspor undateables seizoen rufu...,documentaire voetspor undateables seizoen rufu...,mooi lang nieuw nepnieuws ander voorafgaand Am...,landen zien maken hebben zeggen willen tegenga...,6,0.102652,6,journalism and media,"MEDIA, ARTS, CULTURE & SPORTS"
1,jinek,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,['drones'],...over.\nEn onze tennisdames verrasten de wer...,tennisdame wereld baan klap sensatie nieuws po...,tennisdame wereld baan klap sensatie nieuws po...,regelrecht ander vorig Nederlands internationa...,verrasten meppen beslissen winnen bekennen opl...,0,0.136302,0,war and politics,AI WARFARE & MILIRTARY CONFLICTS
2,goedemorgen nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"['ai', 'kunstmatige intelligentie']",...en de neus van de politie.\nWeet u hoe een ...,neus politie drugslab mens aanraking drug init...,neus politie drugslab mens aanraking drug init...,goed bewuster kunstmatig welmoed hoog laag dir...,weten werken weten denken komen vinden maken z...,19,1.000000,19,health and science,SCIENCE & HEALTH
3,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"['twitter', 'kunstmatige intelligentie']",...zegt de Britse minister van buitenlandse za...,minister zaak aanval aanval gemeenschap aanval...,minister zaak aanval aanval gemeenschap aanval...,Brits buitenlands roekeloos serieus internatio...,zeggen zeggen nemen doen nemen doen veroordele...,1,0.099202,1,public discourse and societal trends,SOCIETY
4,goedemorgen nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,['drone'],...er aardig uit.\nIs het een quarantainecoup?...,quarantainecoup filmp laag respect kappersvak ...,quarantainecoup filmp laag respect kappersvak ...,aardig diep moeilijk heel vast flink droog fli...,gekeken proberen opkrijgen doen nemen willen b...,5,1.000000,5,film industry,"MEDIA, ARTS, CULTURE & SPORTS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,de wereld draait door,2017,"888\nGeerte Piening, Bahar Goodarzi, Jeroen Wo...",yes,"['facebook', 'twitter', 'robot']",...de Staten-Generaal' en niet 'mongool'.\nLev...,staten-generaal mongool staten-generaal lid st...,staten-generaal mongool staten-generaal lid st...,lang ver heel lang gewoon gewoon bezig nieuw b...,leven leven uitmaken gaan lijken rot willen zi...,3,0.072801,3,robots and education,EDUCATION
491,goedemorgen nederland,2022,"888\nGoedemorgen Nederland, welkom terug bij W...",yes,"['drone', 'drones']",...prijsgeven...\nmaar wil aan buitenlandse bo...,bondgenoot topbedrijf land veiligheidsbeurs ve...,bondgenoot topbedrijf land veiligheidsbeurs ve...,buitenlands goed Nederlands Caroline één norma...,prijsgeven willen laten zien verzamelen zijn g...,0,1.000000,0,war and politics,AI WARFARE & MILIRTARY CONFLICTS
492,goedemorgen nederland,2018,"888\nGoedemorgen Nederland, het is vandaag vri...",yes,"['twitter', 'drones']",...Zo gaat zij.\nHeel vrolijk.\nAnneke Nieuwen...,nieuwenhuize jurk hond meid mens foto mailadre...,nieuwenhuize jurk hond meid mens foto mailadre...,heel vrolijk klaar klaar leuk fris bewolkt and...,gaan streken zijn staan vinden zien blijven st...,7,1.000000,7,drones,TECH & AI DEVELOPMENT
493,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"['ai', 'drones']",...weten het niet.\nMaar zij willen iets doen....,dank toelichting nieuws tafel vooruitgang zoek...,dank toelichting nieuws tafel vooruitgang zoek...,hartelijk gauw goed vreselijk stil goed techno...,weten willen doen laten hopen komen zeggen hop...,7,1.000000,7,drones,TECH & AI DEVELOPMENT
